# Mario Kart Tracker v2: Data Scraping

## Tracks

The aim of this notebook is to scrape the tracks from Mario Kart 8 Deluxe and World and store them in a CSV file

Website(s) to scrape from: 
* Mario Wiki

Required Libraries:
* `Selenium`
* `BeautifulSoup`
* `Pandas`

In [2]:
from bs4 import BeautifulSoup
from selenium import webdriver
import pandas as pd
from pathlib import Path
import re

In [3]:
wd = Path().cwd()
wd = wd.parent

### Mario Kart 8 Deluxe

In [4]:
url = 'https://www.mariowiki.com/Mario_Kart_8_Deluxe'

def get_track_html(url):
    driver = webdriver.Firefox()
    driver.get(url)

    # The courses only appear when you scroll down a bit
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')

    track_html = driver.page_source

    driver.quit()

    return track_html


standard_html = get_track_html(url)
print(standard_html)

<html class="client-js" lang="en" dir="ltr"><head>
<meta charset="UTF-8">
<title>Mario Kart 8 Deluxe - Super Mario Wiki, the Mario encyclopedia</title>
<script async="" type="text/javascript" src="https://cmp.inmobi.com/tcfv2/cmp2.js?referer=www.mariowiki.com"></script><script async="" type="text/javascript" src="https://cmp.inmobi.com/choice/v0NnnH1M4W081/www.mariowiki.com/choice.js?tag_version=V3"></script><script>document.documentElement.className="client-js";RLCONF={"wgBreakFrames":false,"wgSeparatorTransformTable":["",""],"wgDigitTransformTable":["",""],"wgDefaultDateFormat":"mdy","wgMonthNames":["","January","February","March","April","May","June","July","August","September","October","November","December"],"wgRequestId":"f93e1cd4e3d8f26480d5b6e4","wgCSPNonce":false,"wgCanonicalNamespace":"","wgCanonicalSpecialPageName":false,"wgNamespaceNumber":0,"wgPageName":"Mario_Kart_8_Deluxe","wgTitle":"Mario Kart 8 Deluxe","wgCurRevisionId":5403865,"wgRevisionId":5403865,"wgArticleId":2200

In [5]:
# We're looking for tables in the "Courses" section
# They'll be in table rows where the first cell contains "Cup"
track_soup = BeautifulSoup(standard_html)

section_name = track_soup.find("span", {"id": "Courses"})
section_head = section_name.find_parent("h2")
standard_courses_table = section_head.find_next_sibling("table")
standard_course_cells = standard_courses_table.find_all("td")

print(standard_course_cells[:2])

[<td><a class="image" href="/File:MK8_MushroomCup.png"><img alt="Mushroom Cup emblem for Mario Kart 8" data-file-height="122" data-file-width="122" decoding="async" height="60" loading="lazy" src="https://mario.wiki.gallery/images/thumb/1/15/MK8_MushroomCup.png/60px-MK8_MushroomCup.png" srcset="https://mario.wiki.gallery/images/thumb/1/15/MK8_MushroomCup.png/90px-MK8_MushroomCup.png 1.5x, https://mario.wiki.gallery/images/thumb/1/15/MK8_MushroomCup.png/120px-MK8_MushroomCup.png 2x" width="60"/></a><br/><a href="/Mushroom_Cup" title="Mushroom Cup">Mushroom Cup</a>
</td>, <td width="21.25%"><a class="image" href="/File:MK8D_Mario_Kart_Stadium_Course_Icon_Full.png"><img alt="The course icon for Mario Kart Stadium in Mario Kart 8 Deluxe" class="sprite" data-file-height="162" data-file-width="288" decoding="async" height="84" loading="lazy" src="https://mario.wiki.gallery/images/thumb/5/50/MK8D_Mario_Kart_Stadium_Course_Icon_Full.png/150px-MK8D_Mario_Kart_Stadium_Course_Icon_Full.png" srcse

In [6]:
from bs4.element import Tag

Some tracks have different names in British English. These are the ones Australia uses, so we need to account for this.

Let's keep both names just to make sure.

In [7]:
def get_british_name(sup: Tag):
    # Get the <a> container, then get the hyperlink id
    id = sup.find("a").get("href")
    # Remove the leading # so we can use it in BeautifulSoup
    id = id.removeprefix("#")
    # Find the element by the id
    note = track_soup.find(id = id)
    # Extract the text from the contained span
    ref_text = note.find("span", class_ = "reference-text")
    ref_text = ref_text.text

    # Extract from between quotation marks
    name = re.findall(r'"(.+)"', ref_text)[0]

    return name

def extract_name(cell: Tag):
    names = {
        "american": None,
        "british": None
    }
    
    # If the name contains the name of the cup, we don't want that, so don't update
    if "Cup" not in cell.text:
        american_name = cell.text.strip("\n")
        
        # Remove any footnote text
        american_name = re.sub(r"\[.\]", "", american_name)
        names["american"] = american_name
        names["british"] = names["american"]

    # We need to check if there is a "sup" element in the cell to check for a footnote
    sup = cell.find_all("sup")

    if sup:
        # If there is, this is the British name, so we need to extract that
        names["british"] = get_british_name(sup[0])

    return names

standard_course_names = [extract_name(cell) for cell in standard_course_cells]

# Remove Nones
standard_course_names = [name for name in standard_course_names if name["american"]]

print(f"{len(standard_course_names)} courses found")

48 courses found


In [8]:
section_name = track_soup.find(id = "Booster_Course_Pass")
section_head = section_name.find_parent("h3")
dlc_courses_table = section_head.find_next("table")
dlc_course_cells = dlc_courses_table.find_all("td")

dlc_names = [extract_name(cell) for cell in dlc_course_cells]

# Remove Nones
dlc_names = [name for name in dlc_names if name]

print(f"{len(dlc_names)} DLC cources found")


48 DLC cources found


In [9]:
# We've now found all the names for MK8!
# Just need to put them into their own dataframe
names = []

names.extend(standard_course_names)
names.extend(dlc_names)

track_df = pd.DataFrame(names)

track_df

,american,british
0,Mario Kart Stadium,Mario Kart Stadium
1,Water Park,Water Park
2,Sweet Sweet Canyon,Sweet Sweet Canyon
3,Thwomp Ruins,Thwomp Ruins
4,Mario Circuit,Mario Circuit
...,...,...
91,Piranha Plant Cove,Piranha Plant Cove
92,Tour Madrid Drive,Tour Madrid Drive
93,3DS Rosalina's Ice World,3DS Rosalina's Ice World
94,SNES Bowser Castle 3,SNES Bowser Castle 3


In [ ]:
# The only thing we have to do is fix the console names. We want the names to appear
# in the format "<Track Name> (<Console>)"
# 3DS has to be before DS, since DS is contained within 3DS
consoles = ["SNES", "N64", "GCN", "GBA", "3DS", "DS", "Wii", "3DS", "Tour"]

def fix_console(track_name: str, consoles: list):
    new_name = track_name
    for console_name in consoles:
        if console_name in new_name:
            # Format the name
            new_name = new_name.replace(console_name, "").strip() + f" ({console_name})"

            break

    return new_name

track_df["db_name"] = track_df["british"].apply(lambda name: fix_console(name, consoles))

track_df

,american,british,db_name
0,Mario Kart Stadium,Mario Kart Stadium,Mario Kart Stadium
1,Water Park,Water Park,Water Park
2,Sweet Sweet Canyon,Sweet Sweet Canyon,Sweet Sweet Canyon
3,Thwomp Ruins,Thwomp Ruins,Thwomp Ruins
4,Mario Circuit,Mario Circuit,Mario Circuit
...,...,...,...
91,Piranha Plant Cove,Piranha Plant Cove,Piranha Plant Cove
92,Tour Madrid Drive,Tour Madrid Drive,Madrid Drive (Tour)
93,3DS Rosalina's Ice World,3DS Rosalina's Ice World,Rosalina's Ice World (3DS)
94,SNES Bowser Castle 3,SNES Bowser Castle 3,Bowser Castle 3 (SNES)


### Mario Kart World

Now we can move onto MK World - hopefully it's as easy as 8

In [30]:
url = 'https://www.ign.com/wikis/mario-kart-world/Track_List'

driver = webdriver.Firefox()
driver.get(url)

world_html = driver.page_source

driver.quit()

print(world_html)

<html lang="en" data-build-id="KU_FytxpY_ELBBZM5olQ_" data-kraken-env="production" data-release="v0.97.49" data-theme="dark" color-scheme="light" style="color-scheme: dark;"><head prefix="og: http://ogp.me/ns# article: http://ogp.me/ns/article# fb: http://www.facebook.com/2008/fbml"><script async="" src="https://www.googletagmanager.com/gtm.js?id=GTM-K7G5DNRH"></script><script src="https://s0.2mdn.net/instream/video/client.js" async="" type="text/javascript"></script><script async="" data-jsonpid="" src="https://cdn-gl.imrworldwide.com/novms/js/2/nlsSDK600.bundle.min.js"></script><script async="" src="https://cdn-gl.imrworldwide.com/conf/config250.js#name=v60Bsdk__1779352521778&amp;ns=NOLBUNDLE"></script><script async="" src="https://sb.scorecardresearch.com/beacon.js"></script><script async="" 0="h" 1="t" 2="t" 3="p" 4="s" 5=":" 6="/" 7="/" 8="b" 9="-" 10="c" 11="o" 12="d" 13="e" 14="." 15="l" 16="i" 17="a" 18="d" 19="m" 20="." 21="c" 22="o" 23="m" 24="/" 25="a" 26="-" 27="0" 28="1" 2

In [40]:
# This page has the tracks in an unordered list so let's search for that!
world_soup = BeautifulSoup(world_html)

list_items = world_soup.find_all('li')

# We want the ones that do not contain any tags within them
list_items = [tag.text for tag in list_items if len(tag.find_all()) == 0]

# Remove duplicates
list_items = list(set(list_items))

list_items

['DK Pass',
 'Shy Guy Bazaar',
 'Koopa Troopa Beach',
 'Faraway Oasis',
 'Crown City',
 "Wario's Galleon",
 'Mario Bros. Circuit',
 'Great\xa0? Block Ruins',
 'Mario Circuit',
 'Peach Stadium (Version 1)',
 'Choco Mountain',
 'DK Spaceport',
 'Peach Stadium (Version 2)',
 'Sky-High Sundae',
 "Bowser's Castle",
 'Salty Salty Speedway',
 'Acorn Heights',
 'Cheep Cheep Falls',
 'Airship Fortress',
 'Boo Cinema',
 'Dry Bones Burnout',
 'Crown City (Version 2)',
 'Moo Moo Meadows',
 'Rainbow Road',
 'Starview Peak',
 'Dandelion Depths',
 'Desert Hills',
 'Peach Beach',
 'Peach Stadium',
 'Crown City (Version 1)',
 "Toad's Factory",
 'Whistlestop Summit',
 'Wario Galleon',
 'Dino Dino Jungle',
 'Wario Stadium']

We have all the tracks, but there are a couple of funny ones which we need to clean up

In [47]:
world_raw_df = pd.DataFrame(list_items, columns=['name'])

# Seems some had a couple of versions, so remove version numbers
world_raw_df['name'] = world_raw_df['name'].str.replace(pat=r' \(Version \d\)', repl='', regex=True)

world_raw_df = world_raw_df.drop_duplicates().sort_values(by='name')

print(f"{world_raw_df.shape[0]} tracks found")
world_raw_df

31 tracks found


,name
16,Acorn Heights
18,Airship Fortress
19,Boo Cinema
14,Bowser's Castle
17,Cheep Cheep Falls
10,Choco Mountain
4,Crown City
0,DK Pass
11,DK Spaceport
25,Dandelion Depths


In [51]:
# There's one double-up due to a misspelling. So we'll just delete it
world_clean_df = world_raw_df.copy()

world_clean_df = world_clean_df[world_clean_df['name'] != 'Wario Galleon']

world_clean_df = world_clean_df.reset_index(drop=True)

print(f"{world_clean_df.shape[0]} tracks found")
world_clean_df

30 tracks found


,name
0,Acorn Heights
1,Airship Fortress
2,Boo Cinema
3,Bowser's Castle
4,Cheep Cheep Falls
5,Choco Mountain
6,Crown City
7,DK Pass
8,DK Spaceport
9,Dandelion Depths


### Saving

Now we have all the tracks, so we can save them!

In [53]:
mk8_track_df = track_df.copy()
world_track_df = world_clean_df.copy()

mk8_track_df['game_version'] = 'Mario Kart 8 Deluxe'
world_track_df['game_version'] = 'Mario Kart World'

all_track_df = pd.concat([mk8_track_df, world_track_df], axis=0)

all_track_df

,name,game_version
0,Excitebike Arena,Mario Kart 8 Deluxe
1,Electrodrome,Mario Kart 8 Deluxe
2,Water Park,Mario Kart 8 Deluxe
3,Toad Harbor,Mario Kart 8 Deluxe
4,SNES Donut Plains 3,Mario Kart 8 Deluxe
...,...,...
25,Starview Peak,Mario Kart World
26,Toad's Factory,Mario Kart World
27,Wario Stadium,Mario Kart World
28,Wario's Galleon,Mario Kart World


In [54]:
all_track_df.to_csv(wd / 'data' / 'tracks.csv', index=False)